# Machine Learning Modeling, Evaluation & MLflow Tracking

## Project
**Mathematical Fingerprint of Musical Success**

## Objectives
- Train a **Linear Regression** model to predict song popularity.
- Train a **Classification** model (Random Forest) to predict hit status.
- Use **MLflow** for experiment tracking and model versioning.
- Perform **Feature Importance** analysis to decode the "formula for success".
- Evaluate models using advanced metrics (Confusion Matrix, F1-score, RMSE).

---

## Data Setup (Required)
1. Open the **Files** pane (folder icon on the left).
2. Ensure the folder named `data` exists.
3. **Upload** the final processed file from ***math_music_eda_feature_engineering.ipynb***:
   - `data/master_dataset_final.csv`

In [ ]:
# ============================================================
# Topic: Environment Setup
# Goal: Install MLflow and import necessary ML libraries
# ============================================================

# Install MLflow (must be done every session in Colab)
!pip install mlflow -q

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# ML libraries
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.linear_model import LinearRegression
from sklearn.ensemble import RandomForestClassifier, RandomForestRegressor
from sklearn.metrics import (accuracy_score, classification_report,
                             confusion_matrix, mean_absolute_error,
                             mean_squared_error, r2_score)

# experiment Tracking
import mlflow

print("Libraries and MLflow installed successfully.")

In [ ]:
# ==============================================================================
# Topic: Data Loading
# Goal: Load the processed dataset from math_music_eda_feature_engineering.ipynb
# ==============================================================================
import os

processed_file = "data/master_dataset_final.csv"

if os.path.exists(processed_file):
    df_model = pd.read_csv(processed_file)
    print(f"Data loaded successfully: {df_model.shape[0]} rows, {df_model.shape[1]} columns.")

    # quick sanity check for the newly added features from math_music_eda_feature_engineering.ipynb
    required_new_cols = ['energy_loudness_ratio', 'mood_index', 'cluster', 'PC1', 'PC2']
    for col in required_new_cols:
        if col in df_model.columns:
            print(f"   - Feature '{col}' found")
        else:
            print(f"   - Feature '{col}' MISSING! Check math_music_eda_feature_engineering.ipynb output.")

    display(df_model.head(3))
else:
    print(f"ERROR: {processed_file} not found in 'data/' folder. Please upload it.")

In [ ]:
# ============================================================
# Topic: Data Preparation
# Goal: Define feature columns and target variables for
#       classification and regression
# ============================================================

# ------------------------------------------------------------
# Feature selection for modeling
# Keep only numerical / engineered music-related features
# Exclude text/id columns and leakage-prone columns
# ------------------------------------------------------------

feature_cols = [
    "duration_ms",
    "explicit",
    "danceability",
    "energy",
    "key",
    "loudness",
    "mode",
    "speechiness",
    "acousticness",
    "instrumentalness",
    "liveness",
    "valence",
    "tempo",
    "time_signature",
    "energy_loudness_ratio",
    "mood_index",
    "PC1",
    "PC2",
    "cluster"
]

# ------------------------------------------------------------
# Convert boolean column to integer if needed
# ------------------------------------------------------------
if df_model["explicit"].dtype == "bool":
    df_model["explicit"] = df_model["explicit"].astype(int)

# ------------------------------------------------------------
# define X and targets
# classification target: is_hit
# regression target: popularity
# ------------------------------------------------------------
X = df_model[feature_cols]

y_class = df_model["is_hit"]
y_reg = df_model["popularity"]

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------
print("Feature matrix and targets created successfully.")
print(f"Number of features: {len(feature_cols)}")
print(f"Feature matrix shape: {X.shape}")
print(f"Classification target shape: {y_class.shape}")
print(f"Regression target shape: {y_reg.shape}")

print("\nSelected features:")
print(feature_cols)

print("\nTarget distribution (is_hit):")
print(y_class.value_counts())

print("\nPopularity summary:")
display(y_reg.describe())

In [ ]:
# mount Google Drive to the Colab environment
# this allows access to files stored in personal Google Drive folders
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
 ============================================================
# Topic: Data Splitting
# Goal:
#   1. Divide the dataset into Training (80%) and Testing (20%) sets
#   2. Use 'stratify' for classification to maintain the rare hit ratio (2.6%)
#   3. Ensure reproducibility using a fixed random_state
# ============================================================

from sklearn.model_selection import train_test_split

# splitting for classification
# goal: train the model on 80% of the songs, and on the remaining 20% ​check whether it recognizes the hits
# use 'stratify' to avoid having no hits in the test due to their rarity
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X, y_class, test_size=0.2, random_state=42, stratify=y_class
)

# splitting for regression
# objective: Preparing data to predict the numerical value of 'popularity'
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X, y_reg, test_size=0.2, random_state=42
)

print("Data splitting complete.")
print(f"Classification Train/Test: {len(X_train_cls)} / {len(X_test_cls)}")
print(f"Regression Train/Test:     {len(X_train_reg)} / {len(X_test_reg)}")

# check hit ratio consistency
hit_ratio_test = (y_test_cls.sum() / len(y_test_cls)) * 100
print(f"Hit ratio in Test set: {hit_ratio_test:.2f}% (Consistent with master data)")